# 🌌 NeoWatch — Phase 2 & 3: EDA, Feature Engineering, SMOTE & Model Training

**Objectives**:
1. **Phase 2 (Checkpoint 2)**: Perform Outlier & Skewness Analysis (IQR), Correlation Heatmap, Scaling (`StandardScaler`), and handle class imbalance using SMOTE. Export `data/processed_asteroid_data.csv` and `models/scaler.pkl`.
2. **Phase 3 (Checkpoint 3)**: Train & Benchmark Baseline Models (Logistic Regression, Random Forest, LightGBM, XGBoost) with 5-Fold Cross Validation. Optimize hyperparameters prioritizing **Recall** and **ROC-AUC**. Save `models/asteroid_model.pkl`.

In [ ]:
import sys
import os
from pathlib import Path

project_root = Path(os.path.abspath('')).parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from src.config import RAW_DATA_PATH, PROCESSED_DATA_PATH, SCALER_PATH, MODEL_PATH, FEATURE_COLUMNS, TARGET_COLUMN
from src.preprocessor import AsteroidPreprocessor
from src.model_trainer import AsteroidModelTrainer

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
print("Environment initialized successfully!")

## 1. Load Raw Asteroid Dataset

In [ ]:
df_raw = pd.read_csv(RAW_DATA_PATH)
print(f"Raw dataset shape: {df_raw.shape}")
df_raw.head()

## 2. Exploratory Data Analysis (EDA)
### 2.1 Target Class Imbalance

In [ ]:
plt.figure(figsize=(6, 4))
sns.countplot(data=df_raw, x=TARGET_COLUMN, palette=['#3498db', '#e74c3c'])
plt.title('Target Variable Distribution (0: Safe, 1: Hazardous)')
plt.xticks([0, 1], ['Non-Hazardous (Safe)', 'Potentially Hazardous (PHA)'])
plt.show()

print(df_raw[TARGET_COLUMN].value_counts(normalize=True) * 100)

### 2.2 Numerical Distributions & Outliers (Boxplots & IQR)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
axes = axes.flatten()

for i, col in enumerate(FEATURE_COLUMNS):
    if col in df_raw.columns:
        sns.boxplot(data=df_raw, y=col, x=TARGET_COLUMN, ax=axes[i], palette=['#3498db', '#e74c3c'])
        axes[i].set_title(f'Distribution of {col}')

plt.tight_layout()
plt.show()

### 2.3 Feature Correlation Heatmap

In [ ]:
plt.figure(figsize=(10, 7))
corr = df_raw[FEATURE_COLUMNS + [TARGET_COLUMN]].corr()
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
plt.title('Feature Correlation Heatmap')
plt.show()

## 3. Preprocessing, Scaling & SMOTE Balancing (Checkpoint 2)

In [ ]:
preprocessor = AsteroidPreprocessor(scaler_type='standard')
X_train, y_train, X_test, y_test, df_processed = preprocessor.prepare_datasets(
    raw_csv_path=RAW_DATA_PATH,
    test_size=0.2,
    random_state=42,
    apply_smote=True
)

print(f"X_train resampled shape: {X_train.shape}")
print(f"X_test scaled shape     : {X_test.shape}")
print(f"\n🎯 Checkpoint 2 Complete! Saved scaler to {SCALER_PATH} and processed data to {PROCESSED_DATA_PATH}")

## 4. Benchmark Models & 5-Fold Cross Validation

In [ ]:
trainer = AsteroidModelTrainer(random_state=42)
df_bench = trainer.run_benchmarks(X_train, y_train, cv_splits=5)
df_bench

## 5. Hyperparameter Tuning for XGBoost (Recall Focused)

In [ ]:
best_xgb = trainer.tune_xgboost(X_train, y_train)

## 6. Test Set Evaluation & Checkpoint 3

In [ ]:
eval_res = trainer.evaluate_model(best_xgb, X_test, y_test)

# Confusion Matrix Plot
plt.figure(figsize=(5, 4))
sns.heatmap(eval_res['confusion_matrix'], annot=True, fmt='d', cmap='Blues',
            xticklabels=['Safe', 'Hazardous'], yticklabels=['Safe', 'Hazardous'])
plt.title('Test Set Confusion Matrix')
plt.ylabel('Actual Label')
plt.xlabel('Predicted Label')
plt.show()

# Save Model Artifact
trainer.save_model(MODEL_PATH)
print(f"\n🎯 Checkpoint 3 Complete! Saved best model to {MODEL_PATH}")